# 可选实验：Logistic Regression 的 Gradient Descent


## 目标

在本实验中，你将：

- 修改 gradient descent，使其适用于 logistic regression。
- 在熟悉的数据集上探索 gradient descent 的表现。


In [ ]:
import copy, math
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
from lab_utils_common import  dlc, plot_data, plt_tumor_data, sigmoid, compute_cost_logistic
from plt_quad_logistic import plt_quad_logistic, plt_prob
plt.style.use('./deeplearning.mplstyle')

## 数据集

我们继续使用 decision boundary 实验中包含两个 feature 的数据集。


In [ ]:
X_train = np.array([[0.5, 1.5], [1,1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])
y_train = np.array([0, 0, 0, 1, 1, 1])

与之前一样，使用辅助函数绘制数据：label $y=1$ 的样本用红色叉号表示，label $y=0$ 的样本用蓝色圆圈表示。


In [ ]:
fig,ax = plt.subplots(1,1,figsize=(4,4))
plot_data(X_train, y_train, ax)

ax.axis([0, 4, 0, 3.5])
ax.set_ylabel('$x_1$', fontsize=12)
ax.set_xlabel('$x_0$', fontsize=12)
plt.show()

## Logistic Regression 的 Gradient Descent

<img align="right" src="./images/C1_W3_Logistic_gradient_descent.png" alt="Logistic regression 的 gradient descent" style="width:400px;padding:10px;">

Gradient descent 利用 gradient 反复更新参数，直到收敛：

$$
\begin{aligned}
w_j &:= w_j-\alpha\frac{\partial J(\mathbf{w},b)}{\partial w_j},\qquad j=0,\ldots,n-1, \\
b &:= b-\alpha\frac{\partial J(\mathbf{w},b)}{\partial b}.
\end{aligned}
\tag{1}
$$

每次迭代都必须**同时更新**全部 $w_j$ 和 $b$，其中：

$$
\frac{\partial J(\mathbf{w},b)}{\partial w_j}=\frac{1}{m}\sum_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})-y^{(i)}\right)x_j^{(i)} \tag{2}
$$

$$
\frac{\partial J(\mathbf{w},b)}{\partial b}=\frac{1}{m}\sum_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})-y^{(i)}\right) \tag{3}
$$

- $m$ 是训练样本数。
- $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ 是预测值，$y^{(i)}$ 是 target。
- 对于 logistic regression，$z=\mathbf{w}\cdot\mathbf{x}+b$，$f_{\mathbf{w},b}(\mathbf{x})=g(z)$，其中 $g(z)=\frac{1}{1+e^{-z}}$ 是 sigmoid function。


### 实现 Gradient Descent

实现包括两部分：

- 按公式 (1) 进行循环更新，即下面的 `gradient_descent`。可选实验和编程练习通常会提供这个函数。
- 按公式 (2)、(3) 计算当前 gradient，即下面的 `compute_gradient_logistic`。本周的编程练习会要求你实现这一部分。

#### Gradient 计算的代码说明

下面为所有 $w_j$ 和 $b$ 实现公式 (2)、(3)：

1. 初始化用于累加的 `dj_dw` 和 `dj_db`。
2. 遍历每个样本，计算误差 $g(\mathbf{w}\cdot\mathbf{x}^{(i)}+b)-y^{(i)}$。
3. 遍历该样本的各个输入 $x_j^{(i)}$，将误差乘以 $x_j^{(i)}$，累加到 `dj_dw` 的对应元素中。
4. 将该样本的误差累加到 `dj_db`。
5. 遍历结束后，将 `dj_dw` 和 `dj_db` 分别除以样本总数 $m$。

在 NumPy 中，$\mathbf{x}^{(i)}$ 对应 `X[i, :]` 或 `X[i]`，$x_j^{(i)}$ 对应 `X[i, j]`。


In [ ]:
def compute_gradient_logistic(X, y, w, b): 
    """
    Computes the gradient for logistic regression 
 
    Args:
      X (ndarray (m,n): Data, m examples with n features
      y (ndarray (m,)): target values
      w (ndarray (n,)): model parameters  
      b (scalar)      : model parameter
    Returns
      dj_dw (ndarray (n,)): The gradient of the cost w.r.t. the parameters w. 
      dj_db (scalar)      : The gradient of the cost w.r.t. the parameter b. 
    """
    m,n = X.shape
    dj_dw = np.zeros((n,))                           #(n,)
    dj_db = 0.

    for i in range(m):
        f_wb_i = sigmoid(np.dot(X[i],w) + b)          #(n,)(n,)=scalar
        err_i  = f_wb_i  - y[i]                       #scalar
        for j in range(n):
            dj_dw[j] = dj_dw[j] + err_i * X[i,j]      #scalar
        dj_db = dj_db + err_i
    dj_dw = dj_dw/m                                   #(n,)
    dj_db = dj_db/m                                   #scalar
        
    return dj_db, dj_dw  

运行下面的代码单元，检查 gradient 函数的实现。


In [ ]:
X_tmp = np.array([[0.5, 1.5], [1,1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])
y_tmp = np.array([0, 0, 0, 1, 1, 1])
w_tmp = np.array([2.,3.])
b_tmp = 1.
dj_db_tmp, dj_dw_tmp = compute_gradient_logistic(X_tmp, y_tmp, w_tmp, b_tmp)
print(f"dj_db: {dj_db_tmp}" )
print(f"dj_dw: {dj_dw_tmp.tolist()}" )

**预期输出：**

```text
dj_db: 0.49861806546328574
dj_dw: [0.498333393278696, 0.49883942983996693]
```


### Gradient Descent 代码

下面的代码实现公式 (1)。请找出函数中的各个步骤，并与上面的公式对照。


In [ ]:
def gradient_descent(X, y, w_in, b_in, alpha, num_iters): 
    """
    Performs batch gradient descent
    
    Args:
      X (ndarray (m,n)   : Data, m examples with n features
      y (ndarray (m,))   : target values
      w_in (ndarray (n,)): Initial values of model parameters  
      b_in (scalar)      : Initial values of model parameter
      alpha (float)      : Learning rate
      num_iters (scalar) : number of iterations to run gradient descent
      
    Returns:
      w (ndarray (n,))   : Updated values of parameters
      b (scalar)         : Updated value of parameter 
    """
    # An array to store cost J and w's at each iteration primarily for graphing later
    J_history = []
    w = copy.deepcopy(w_in)  #avoid modifying global w within function
    b = b_in
    
    for i in range(num_iters):
        # Calculate the gradient and update the parameters
        dj_db, dj_dw = compute_gradient_logistic(X, y, w, b)   

        # Update Parameters using w, b, alpha and gradient
        w = w - alpha * dj_dw               
        b = b - alpha * dj_db               
      
        # Save cost J at each iteration
        if i<100000:      # prevent resource exhaustion 
            J_history.append( compute_cost_logistic(X, y, w, b) )

        # Print cost every at intervals 10 times or as many iterations if < 10
        if i% math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Cost {J_history[-1]}   ")
        
    return w, b, J_history         #return final w,b and J history for graphing


下面在数据集上运行 gradient descent。


In [ ]:
w_tmp  = np.zeros_like(X_train[0])
b_tmp  = 0.
alph = 0.1
iters = 10000

w_out, b_out, _ = gradient_descent(X_train, y_train, w_tmp, b_tmp, alph, iters) 
print(f"\nupdated parameters: w:{w_out}, b:{b_out}")

### 绘制 Gradient Descent 的结果


In [ ]:
fig,ax = plt.subplots(1,1,figsize=(5,4))
# plot the probability 
plt_prob(ax, w_out, b_out)

# Plot the original data
ax.set_ylabel(r'$x_1$')
ax.set_xlabel(r'$x_0$')   
ax.axis([0, 4, 0, 3.5])
plot_data(X_train,y_train,ax)

# Plot the decision boundary
x0 = -b_out/w_out[0]
x1 = -b_out/w_out[1]
ax.plot([0,x0],[x1,0], c=dlc["dlblue"], lw=1)
plt.show()

在上图中：

- 背景颜色表示模型预测的 $y=1$ 的概率，即应用分类 threshold 之前的输出。
- Decision boundary 是预测概率为 0.5 的位置组成的直线。


## 另一个数据集

下面回到单变量数据集。由于只有 $w$ 和 $b$ 两个参数，可以绘制 cost function 的 contour plot，更直观地观察 gradient descent 的过程。


In [ ]:
x_train = np.array([0., 1, 2, 3, 4, 5])
y_train = np.array([0,  0, 0, 1, 1, 1])

与之前一样，使用辅助函数绘制数据：label $y=1$ 的样本用红色叉号表示，label $y=0$ 的样本用蓝色圆圈表示。


In [ ]:
fig,ax = plt.subplots(1,1,figsize=(4,3))
plt_tumor_data(x_train, y_train, ax)
plt.show()

在下图中尝试以下操作：

1. 点击右上角 contour plot 中的不同位置，改变 $w$ 和 $b$。图表更新可能需要一两秒。观察左上角 cost 的变化，以及每个样本的 loss 如何共同构成 cost（图中的竖直虚线用于表示各样本的误差）。
2. 点击橙色按钮运行 gradient descent，观察 cost 持续下降。注意，contour plot 和 cost 曲线使用的是 $\log(\text{cost})$。
3. 再次点击 contour plot，可以重设模型并开始新一轮运行。
4. 如需重置整个图表，重新运行代码单元。


In [ ]:
w_range = np.array([-1, 7])
b_range = np.array([1, -14])
quad = plt_quad_logistic( x_train, y_train, w_range, b_range )

## 恭喜完成实验！

你已经查看了 logistic regression 的 gradient 公式及其实现，并使用这些函数分别探索了单变量和双变量数据集。
